# 🏏 Cricket Score Prediction AI Assistant
### Offline RAG System powered by Ollama + FAISS + Mistral

---

## What does this notebook do?

This notebook builds a **fully offline Retrieval-Augmented Generation (RAG)** system that acts as an expert cricket analyst. It can:
- Answer questions about match scores, player statistics, team performance
- Provide context-grounded predictions based on historical cricket data
- Explain pitch conditions, weather impact, and batting/bowling trends

**No API key required. Everything runs locally using Ollama.**

---

## RAG Pipeline Overview

```
  Cricket Dataset (HuggingFace / Built-in corpus)
        |
        v
  [Step 1] Install Dependencies & Start Ollama
        |
        v
  [Step 2] Load Cricket Dataset
        |
        v
  [Step 3] Chunk Documents
        |
        v
  [Step 4] Create Embeddings -> FAISS Vector Store
        |
        v
  [Step 5] Setup Mistral LLM via Ollama
        |
        v
  [Step 6] Build Prompt Template
        |
        v
  [Step 7] Assemble RAG Chain
        |
        v
  [Step 8] Ask Cricket Questions!
        |
        v
  [Step 9] Gradio Chat UI
```

---
## Step 1 — Install Dependencies & Start Ollama
This installs all required libraries and starts the Ollama local LLM server. **Run this cell first.**

In [1]:
!apt-get install -y -q zstd
print("zstd installed successfully")

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 100 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (6,131 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122402 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
zstd installed successfully


In [2]:
!curl -fsSL https://ollama.com/install.sh | sh
print("Ollama installed successfully")

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Ollama installed successfully


In [3]:
!pip install -q ollama faiss-cpu sentence-transformers datasets langchain langchain-community langchain-ollama langchain-huggingface langchain-text-splitters gradio
print("All Python libraries installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
All Python libraries installed


In [18]:
import subprocess
import time
import requests

# Start Ollama server in background
proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("Waiting for Ollama server to start...")

for i in range(30):
    try:
        response = requests.get("http://localhost:11434")
        if response.status_code == 200:
            print("Ollama server is running!")
            break
    except Exception:
        time.sleep(2)
else:
    print("Server did not start - re-run this cell once")

Waiting for Ollama server to start...
Ollama server is running!


In [19]:
print("Pulling Mistral model (~4GB). This may take a few minutes...")
!ollama pull mistral
print("Mistral model ready!")

Pulling Mistral model (~4GB). This may take a few minutes...

Mistral model ready!


In [20]:
import ollama

# Make sure Ollama server is running locally:
# Run this in terminal before executing Python code:
# ollama serve

try:
    test_response = ollama.chat(
        model="mistral",
        messages=[
            {
                "role": "user",
                "content": "In one sentence, what is a cricket innings?"
            }
        ]
    )

    print("Mistral test response:")
    print(test_response["message"]["content"])

except ConnectionError:
    print("Ollama server is not running.")
    print("Start it using: ollama serve")

Mistral test response:
 A cricket innings refers to the total number of runs scored by a team or a batsman during their turn at batting. It consists of multiple overs.


---
## Step 2 — Load the Cricket Dataset

We first attempt to load a cricket-relevant HuggingFace dataset.
If unavailable, a rich built-in corpus of cricket knowledge is used.

The built-in corpus covers:
- Match formats (Test, ODI, T20)
- Famous matches and records
- Batting / bowling statistics
- Pitch & weather impact on scores
- Score prediction factors
- Team strategies

In [24]:
import pandas as pd
from datasets import load_dataset

print("Attempting to load cricket dataset from HuggingFace...")
DATASET_LOADED = False

try:
    ds = load_dataset("nirmalkumar/cricket-commentary", split="train", trust_remote_code=True)
    df = ds.to_pandas()
    text_col = [c for c in df.columns if "text" in c.lower() or "description" in c.lower() or "comment" in c.lower()]
    if text_col:
        df = df[[text_col[0]]].rename(columns={text_col[0]: "text"}).dropna()
        df["text"] = df["text"].astype(str)
        df = df[df["text"].str.len() > 100].head(500).reset_index(drop=True)
        df["doc_id"] = [f"Cricket_Doc_{i}" for i in range(len(df))]
        print(f"Loaded {len(df)} cricket records from HuggingFace")
        DATASET_LOADED = True
except Exception as e:
    print(f"HuggingFace dataset unavailable ({e})")
    print("Falling back to built-in cricket knowledge corpus")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'nirmalkumar/cricket-commentary' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'nirmalkumar/cricket-commentary' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Attempting to load cricket dataset from HuggingFace...


dataset_infos.json:   0%|          | 0.00/945 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.90M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/2.06M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50204 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/27381 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12816 [00:00<?, ? examples/s]

In [25]:
if not DATASET_LOADED:
    cricket_docs = [
        {
            "doc_id": "T20_Score_Prediction_Factors",
            "text": "In T20 cricket, the first-innings average score on flat pitches in home conditions is around 160-180 runs. Key factors affecting score prediction include: powerplay performance (overs 1-6), pitch dryness, dew factor in evening matches, and team's batting depth. Teams batting first historically score 10-15 more runs on average than teams chasing in T20I matches. Dew in the second innings makes the ball slippery, benefiting batters. A loss of 3 wickets in powerplay typically caps scores under 150."
        },
        {
            "doc_id": "ODI_Score_Prediction_Factors",
            "text": "ODI cricket scores are determined by 50 overs per side. Average first-innings scores range from 250-290 on neutral pitches. Important predictive variables include: the condition of the pitch after 30 overs, swing in the first 10 overs, batting Powerplay (overs 41-50), and middle-over economy rates. In subcontinent conditions (India, Pakistan, Sri Lanka), totals are typically higher (270-300) due to flat pitches. In England or New Zealand, swing bowling keeps totals to 230-260. The Duckworth-Lewis-Stern (DLS) method is used for rain-affected matches."
        },
        {
            "doc_id": "Test_Match_Scoring_Patterns",
            "text": "Test cricket is played over 5 days with teams batting twice each. Day 1 on fresh pitches averages 280-320 runs. Pitch deteriorates by day 3-4, aiding spin bowling. Teams expect 350+ in the first innings on batting-friendly surfaces. A total under 200 is considered low and likely to invite a follow-on (if the deficit is 200+ runs). Teams like India and Australia consistently score 350-450 in Tests. England's Bazball approach post-2022 has pushed Test scoring rates to 4+ runs per over."
        },
        {
            "doc_id": "Virat_Kohli_Batting_Stats",
            "text": "Virat Kohli is one of cricket's greatest run-scorers. He has 8000+ ODI runs at an average of 58, with 46 ODI centuries the most by any batter. In T20Is, he averages 52 with 1 century. His Test average is 48 with 29 centuries. Kohli is known for chasing targets, averaging over 66 while chasing in ODIs. His presence in the batting lineup typically adds 30-40 runs to a team's projected score. Under pressure conditions, his conversion rate from 50 to 100 is among the highest in world cricket."
        },
        {
            "doc_id": "Rohit_Sharma_Opening_Batting",
            "text": "Rohit Sharma is India's T20I captain and an explosive opener. He holds the record for most T20I centuries (4) and has scored 3 ODI double-centuries. In T20Is, Rohit averages 31 with a strike rate of 140, often scoring 70-90 in powerplay when in form. He hit 264 against Sri Lanka in ODIs, the highest individual ODI score. In ICC tournaments, Rohit is particularly dangerous: he averaged 56 in the 2019 World Cup with 5 centuries."
        },
        {
            "doc_id": "Jasprit_Bumrah_Bowling_Stats",
            "text": "Jasprit Bumrah is India's premier fast bowler across all formats. In Test cricket, he averages under 22 with the ball and has taken 150+ wickets. In T20Is, he consistently bowls economy under 7.0 and is India's death-overs specialist. Bumrah's yorker is considered the best in world cricket. In the 2023 IPL, he took 20 wickets for Mumbai Indians. Teams facing Bumrah typically score 15-20% fewer runs in the overs he bowls compared to average fast bowlers."
        },
        {
            "doc_id": "Pitch_Report_Types",
            "text": "Cricket pitch types significantly affect score predictions. Green tops in UK and New Zealand mean pacers dominate, expect low scores 220-250 in Tests. Flat dry pitches in India and Pakistan mean spinners take over in later stages, first-innings scores of 400+ are common. Red soil pitches in Australia have good pace and bounce, batters score freely if they survive early swings. A pitch with grass covering assists swing and seam for first 30 overs; a dusty pitch spins from Day 1 in Test cricket."
        },
        {
            "doc_id": "IPL_Score_Trends",
            "text": "The Indian Premier League (IPL) is the world's richest T20 league. IPL 2024 average first-innings score was 174 runs. At Wankhede Stadium (Mumbai), average scores are 185-200 due to short boundaries. At Chepauk (Chennai), spin-friendly surfaces keep totals under 160. At Chinnaswamy (Bangalore), scores often exceed 190 due to a high-altitude flat pitch. The highest team total in IPL history is 287/2 by Royal Challengers Bangalore. Teams defending under 150 in IPL have only a 30% win rate."
        },
        {
            "doc_id": "ICC_World_Cup_2023_Stats",
            "text": "The ICC Men's Cricket World Cup 2023 was held in India. India won all league matches and averaged 330 runs per innings. Australia defeated India in the final at Ahmedabad by 6 wickets. South Africa and New Zealand also posted consistent 280-300 run totals. The highest individual score was Quinton de Kock's 174 against NZ. The tournament's highest team total was India's 397/4 against Afghanistan. Average powerplay score across the tournament was 57 runs for 1 wicket."
        },
        {
            "doc_id": "T20_World_Cup_2024_Stats",
            "text": "The ICC Men's T20 World Cup 2024 was held in the USA and West Indies. India defeated South Africa by 7 runs in the final. USA pitches were low-scoring, with averages of 130-145. West Indian pitches favored pace bowling. Rohit Sharma was India's highest scorer. India's bowling lineup, led by Bumrah and Arshdeep Singh, conceded under 130 in most group matches. The average first-innings T20 World Cup score in 2024 was 142 runs. Afghanistan's spin attack was the most economical with an average of 6.2 RPO."
        },
        {
            "doc_id": "Chasing_vs_Setting_Strategy",
            "text": "In T20 cricket, teams winning the toss prefer to chase in about 60% of cases globally. In the IPL, the team batting second wins 54% of matches. Chasing is preferred due to the dew factor, knowledge of the target, and the ability to pace the innings. When a target is under 150, defending teams win 70% of the time. Over 180, chasing teams need a strong top-order: if the top 3 score 130+, chase success rate is 85%. Teams with batting depth (hitting up to No. 8) are better chasers."
        },
        {
            "doc_id": "Weather_Dew_Impact_Cricket",
            "text": "Weather profoundly affects cricket scores. Overcast conditions assist swing bowling, keeping scores low in the morning session. Bright sunshine dries the pitch and helps batters. Dew in evening matches (especially in Asian countries) makes the ball wet and harder to grip for bowlers, assisting batting sides. In the 2023 ODI World Cup held in India's winter, heavy dew in night matches at Pune and Lucknow reduced bowling effectiveness significantly. Rain interruptions invoke the DLS method, which can dramatically alter target scores."
        },
        {
            "doc_id": "Powerplay_Analysis_Cricket",
            "text": "Powerplay overs are crucial for score prediction. In T20 cricket, overs 1-6 are mandatory fielding restrictions. Teams averaging 50+ in powerplay win 65% of T20I matches. Top powerplay scorers globally: Jos Buttler (England) averages 55 runs in T20I powerplays. In ODIs, the first 10 overs set the platform; teams scoring 60+ in the first ODI powerplay average 275 total. Bowlers who strike in the powerplay deflate totals significantly: teams losing 2 wickets in T20 powerplay average only 138 total compared to 172 when they lose 0 wickets."
        },
        {
            "doc_id": "Death_Overs_Scoring_Cricket",
            "text": "Death overs (overs 16-20 in T20, overs 46-50 in ODI) are where scores accelerate dramatically. In IPL, teams score on average 60-70 runs in death overs (overs 16-20). Top death-over batters include Hardik Pandya, MS Dhoni (historical), and Andre Russell who averages a strike rate of 200+ in overs 17-20. Teams with a specialist death bowler like Bumrah concede 15-20 fewer runs in the death compared to the average. In ODIs, teams scoring 60+ in the last 10 overs consistently post 300+ totals."
        },
        {
            "doc_id": "Australia_Test_Cricket_Records",
            "text": "Australia is the most successful Test cricket nation with 400+ wins. At home venues (SCG, MCG, Gabba), Australia averages 380 runs per Test innings. The Gabba in Brisbane was considered an impenetrable fortress: Australia had not lost a Test there since 1988 until India won in January 2021 with a 3-wicket victory chasing 328. Australia's highest Test total is 758/8d. Steve Smith averages 59 in Test cricket, second only to Don Bradman's legendary 99.94."
        },
        {
            "doc_id": "MS_Dhoni_Finishing_Legacy",
            "text": "MS Dhoni is widely regarded as the greatest finisher in cricket history. His ODI average of 50 with a career strike rate of 87 is remarkable for a lower-order batter. Dhoni has a 70%+ win rate when India requires under 50 from the last 5 overs and he is at the crease. His famous six to win the 2011 World Cup final against Sri Lanka sealed India's triumph. As CSK captain in IPL, he won 5 titles. His ability to score off the last ball of an over (hitting sixes off good deliveries) has inspired the finisher role in all T20 cricket worldwide."
        },
        {
            "doc_id": "Spin_Bowling_Impact_Subcontinent",
            "text": "Spin bowling is the dominant force in subcontinental cricket (India, Pakistan, Sri Lanka, Bangladesh). In India's home Tests, spinners take 60-65% of wickets. Ravichandran Ashwin has 500+ Test wickets at home at an average under 24. In dry conditions on the subcontinent, teams posting 250+ on day 1 of a Test are in a winning position 80% of the time because the pitch deteriorates sharply. In T20Is on dusty subcontinent pitches, leg-spinners and off-spinners bowl 40% of overs and concede less than pace bowlers in the middle overs."
        },
        {
            "doc_id": "Head_to_Head_India_vs_Pakistan",
            "text": "India vs Pakistan is the most-watched cricket rivalry in the world. In World Cup matches (ODI + T20), India lead 7-1 overall. In T20 World Cups, India have won both meetings (2022 Melbourne and 2024 New York). Pakistan averages 155 runs when batting first in T20 World Cup matches against India. India's bowling attack has restricted Pakistan to under 160 in 5 of their last 6 T20I encounters. Expected score when India meets Pakistan in a neutral T20 venue: 160-175 first innings."
        },
        {
            "doc_id": "DLS_Method_Explanation",
            "text": "The Duckworth-Lewis-Stern (DLS) method is a mathematical formula used to calculate revised targets in rain-affected limited-overs cricket matches. It takes into account the number of overs remaining and wickets in hand to compute a resource percentage. If a team has faced 20 of 50 overs and is 100/2 with rain stopping play, their total resource used is compared to the interrupted target's resource. The DLS method replaced the Run Rate method in 1997. Key insight: teams with more wickets in hand have greater DLS resources. Losing early wickets dramatically reduces DLS-adjusted par scores."
        },
        {
            "doc_id": "Ben_Stokes_AllRounder_Impact",
            "text": "Ben Stokes is England's Test captain and rated the world's best allrounder. As a batter, his Test average of 36 understates his impact in critical matches: he averages 54 in winning causes. His 135* against Australia in the 2019 World Cup final powered England to a Super Over. As a bowler, he contributes around 30-40 wickets per year in Tests. His aggressive batting style under Bazball has helped England average 4.2 runs per over in recent Tests, the highest in Test history for a sustained period."
        }
    ]
    df = pd.DataFrame(cricket_docs)
    print(f"Built-in cricket knowledge corpus ready: {len(df)} documents")

print("\nDocument IDs available:")
for doc_id in df["doc_id"].tolist():
    print(f"  - {doc_id}")

Built-in cricket knowledge corpus ready: 20 documents

Document IDs available:
  - T20_Score_Prediction_Factors
  - ODI_Score_Prediction_Factors
  - Test_Match_Scoring_Patterns
  - Virat_Kohli_Batting_Stats
  - Rohit_Sharma_Opening_Batting
  - Jasprit_Bumrah_Bowling_Stats
  - Pitch_Report_Types
  - IPL_Score_Trends
  - ICC_World_Cup_2023_Stats
  - T20_World_Cup_2024_Stats
  - Chasing_vs_Setting_Strategy
  - Weather_Dew_Impact_Cricket
  - Powerplay_Analysis_Cricket
  - Death_Overs_Scoring_Cricket
  - Australia_Test_Cricket_Records
  - MS_Dhoni_Finishing_Legacy
  - Spin_Bowling_Impact_Subcontinent
  - Head_to_Head_India_vs_Pakistan
  - DLS_Method_Explanation
  - Ben_Stokes_AllRounder_Impact


---
## Step 3 — Chunk the Documents

Long documents are split into overlapping chunks so the embedding model can represent each chunk meaningfully, and the retriever can fetch precise passages relevant to each question.

In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

splitter = RecursiveCharacterTextSplitter(chunk_size=450, chunk_overlap=80)

documents = []

for _, row in df.iterrows():
    text = row["text"]
    doc_id = row["doc_id"]
    for i, chunk in enumerate(splitter.split_text(text)):
        documents.append(Document(
            page_content=chunk,
            metadata={"source": doc_id, "chunk": i}
        ))

print(f"Chunking complete!")
print(f"  Original documents  : {len(df)}")
print(f"  Total chunks created: {len(documents)}")
print(f"  Avg chunks per doc  : {len(documents) / len(df):.1f}")
print(f"\nSample chunk preview:")
print(f"  Source  : {documents[0].metadata['source']}")
print(f"  Content : {documents[0].page_content[:250]}...")

Chunking complete!
  Original documents  : 20
  Total chunks created: 39
  Avg chunks per doc  : 1.9

Sample chunk preview:
  Source  : T20_Score_Prediction_Factors
  Content : In T20 cricket, the first-innings average score on flat pitches in home conditions is around 160-180 runs. Key factors affecting score prediction include: powerplay performance (overs 1-6), pitch dryness, dew factor in evening matches, and team's bat...


---
## Step 4 — Create Embeddings & Build FAISS Vector Store

Each chunk is converted to a 384-dimensional vector using `all-MiniLM-L6-v2`. FAISS indexes all vectors for fast similarity search at query time.

In [27]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("Loading embedding model on GPU...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Building FAISS vector store from document chunks...")
vectorstore = FAISS.from_documents(documents, embeddings)
vectorstore.save_local("/content/cricket_faiss_index")

print(f"FAISS index ready!")
print(f"  Total vectors indexed: {vectorstore.index.ntotal}")
print(f"  Index saved to       : /content/cricket_faiss_index")

Loading embedding model on GPU...


/tmp/ipykernel_3012/4278076970.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Building FAISS vector store from document chunks...
FAISS index ready!
  Total vectors indexed: 39
  Index saved to       : /content/cricket_faiss_index


---
## Step 5 — Set Up Mistral LLM via Ollama

Mistral runs 100% locally. No API key, no internet calls during inference.

In [28]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="mistral",
    temperature=0.1,   # Low temperature = factual, consistent answers
    num_predict=500    # Max tokens in response
)

print("Mistral LLM configured via Ollama")
print("  Model      : mistral")
print("  Temperature: 0.1 (factual mode)")
print("  Max tokens : 500")

Mistral LLM configured via Ollama
  Model      : mistral
  Temperature: 0.1 (factual mode)
  Max tokens : 500


---
## Step 6 — Build the Cricket Prompt Template

The prompt instructs Mistral to act as a **cricket analyst**, answer only from retrieved documents, and give structured cricket-specific responses with statistics.

In [29]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("""
You are an expert cricket analyst and score prediction assistant.
Your role is to answer questions about cricket matches, player statistics,
pitch conditions, team strategies, and score predictions.

Answer the question using ONLY the cricket documents provided below.
Cite the document names in your answer when referencing specific statistics or matches.
If the documents do not contain enough information to answer, say so clearly.
Be precise with numbers -- always include run rates, averages, and percentages when available.

Cricket Knowledge Documents:
{context}

Question: {question}

Cricket Analysis Answer:""")

print("Cricket prompt template created successfully")

Cricket prompt template created successfully


---
## Step 7 — Assemble the RAG Chain

Wire the retriever (FAISS) + prompt template + Mistral LLM into one callable pipeline.

In [30]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}   # Retrieve top-4 most relevant chunks
)

def format_docs(docs):
    return "\n\n".join(
        f"[{d.metadata['source']}]\n{d.page_content}" for d in docs
    )

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain assembled successfully!")
print("Pipeline: FAISS Retriever (k=4) -> Cricket Prompt -> Mistral -> Answer")

RAG chain assembled successfully!
Pipeline: FAISS Retriever (k=4) -> Cricket Prompt -> Mistral -> Answer


---
## Step 8 — Ask Cricket Questions!

Use the `ask()` helper to query the RAG system. It shows which documents were retrieved and the model's grounded answer.

In [31]:
def ask(question: str):
    print(f"\n{'='*70}")
    print(f"QUESTION: {question}")
    print(f"{'='*70}")

    docs = retriever.invoke(question)
    sources = list(dict.fromkeys(d.metadata["source"] for d in docs))
    print(f"Documents retrieved: {', '.join(sources)}")
    print("-" * 70)

    answer = rag_chain.invoke(question)
    print(f"ANSWER: {answer}")
    return answer

In [ ]:
_ = ask("What is the average first-innings score in T20 cricket and what factors affect it?")

In [ ]:
_ = ask("How does dew affect score prediction in evening T20 matches?")

In [ ]:
_ = ask("What is the predicted score if a team loses 3 wickets in the T20 powerplay?")

In [ ]:
_ = ask("How does Jasprit Bumrah impact the score when he bowls?")

In [ ]:
_ = ask("What is a good target to defend in the IPL and what is the win probability?")

In [ ]:
_ = ask("How does the DLS method work when rain interrupts a match?")

In [ ]:
_ = ask("What score can India expect against Pakistan in a T20 World Cup match?")

In [ ]:
_ = ask("Which IPL stadium produces the highest scores and why?")

---
## Step 9 — Gradio Chat UI

Launch an interactive chat interface for the Cricket Score Prediction AI Assistant. The Gradio `share=True` generates a public link you can open on any device.

In [32]:
import gradio as gr
import ollama

# ── Helper functions ──────────────────────────────────────────────────
def get_sources(docs):
    return list(dict.fromkeys(d.metadata["source"] for d in docs))


def cricket_rag_response(message, history):
    """Full RAG response function for Gradio chatbot."""
    docs = retriever.invoke(message)
    sources = get_sources(docs)
    context = "\n\n".join(
        f"[{d.metadata['source']}]\n{d.page_content}" for d in docs
    )

    prompt_text = f"""You are an expert cricket analyst and score prediction assistant.
Your role is to answer questions about cricket matches, player statistics,
pitch conditions, team strategies, and score predictions.

Answer the question using ONLY the cricket documents provided below.
Cite the document names in your answer when referencing specific statistics or matches.
If the documents do not contain enough information to answer, say so clearly.
Be precise with numbers -- always include run rates, averages, and percentages when available.

Cricket Knowledge Documents:
{context}

Question: {message}

Cricket Analysis Answer:"""

    response = ollama.chat(
        model="mistral",
        messages=[{"role": "user", "content": prompt_text}]
    )
    answer = response["message"]["content"]

    source_line = (
        "\n\n---\n**Sources:** "
        + " | ".join(f"`{s}`" for s in sources)
    )
    return answer + source_line


# ── Gradio UI ─────────────────────────────────────────────────────────
with gr.Blocks(title="Cricket Score Prediction AI", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # Cricket Score Prediction AI Assistant
    **Powered by Mistral + FAISS + Offline RAG**

    Ask questions about cricket scores, player stats, pitch conditions, and match predictions.
    All answers are grounded in cricket knowledge documents.
    """)

    chatbot = gr.Chatbot(
        label="Cricket AI Assistant",
        height=500,
        show_copy_button=True,
        render_markdown=True,
    )

    with gr.Row():
        txt = gr.Textbox(
            placeholder="e.g. What score should India set to win a T20 at Wankhede?",
            show_label=False,
            scale=9,
        )
        btn = gr.Button("Ask", scale=1, variant="primary")

    gr.Examples(
        examples=[
            "What is the average T20 score and what factors affect it?",
            "How does dew impact evening T20 matches?",
            "What score is safe to defend in the IPL?",
            "How does Jasprit Bumrah impact the opponent's score?",
            "How does pitch type affect scoring in Test cricket?",
            "What is India's record against Pakistan in T20 World Cups?",
            "How does DLS method work in rain-affected matches?",
            "What is a good powerplay score in ODI cricket?",
            "Which IPL venue has the highest average score?",
            "How does losing early wickets affect DLS par score?",
        ],
        inputs=txt,
    )

    def respond(message, chat_history):
        answer = cricket_rag_response(message, chat_history)
        chat_history.append((message, answer))
        return "", chat_history

    txt.submit(respond, [txt, chatbot], [txt, chatbot])
    btn.click(respond, [txt, chatbot], [txt, chatbot])

demo.launch(share=True, quiet=True)

/tmp/ipykernel_3012/2184908401.py:47: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Cricket Score Prediction AI", theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_3012/2184908401.py:57: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_3012/2184908401.py:57: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(
/tmp/ipykernel_3012/2184908401.py:57: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. 

* Running on public URL: https://0419ed77aa52325e29.gradio.live


---
## Evaluation — Save Index & Run Benchmark Questions

In [33]:
import os

INDEX_SAVE_PATH = "/content/cricket_faiss_final"
vectorstore.save_local(INDEX_SAVE_PATH)
print(f"Index saved to: {INDEX_SAVE_PATH}/")
print(f"Files: {os.listdir(INDEX_SAVE_PATH)}")
print()

EVALUATION_QUESTIONS = [
    "What is the average first-innings T20 score in the IPL?",
    "What pitch type produces the highest scores in Test cricket?",
    "How many runs does Virat Kohli typically add to India's projected score?",
    "What happens to a T20 total when 3 wickets fall in the powerplay?",
    "What was India's average score in the 2023 ODI World Cup?",
]

print("=" * 70)
print("CRICKET RAG EVALUATION")
print("=" * 70)

for i, question in enumerate(EVALUATION_QUESTIONS, 1):
    print(f"\nQ{i}: {question}")
    print("-" * 55)
    docs = retriever.invoke(question)
    sources = list(dict.fromkeys(d.metadata["source"] for d in docs))
    answer = rag_chain.invoke(question)
    print(f"Answer : {answer.strip()[:400]}")
    print(f"Sources: {', '.join(sources)}")

print("\n" + "=" * 70)
print("Evaluation complete! Review the answers above for accuracy.")
print("Are they grounded in the cricket documents? Are statistics correct?")
print("=" * 70)

Index saved to: /content/cricket_faiss_final/
Files: ['index.pkl', 'index.faiss']

CRICKET RAG EVALUATION

Q1: What is the average first-innings T20 score in the IPL?
-------------------------------------------------------
Answer : The average first-innings T20 score in the IPL, as per the document [IPL_Score_Trends], is 174 runs.
Sources: IPL_Score_Trends, Head_to_Head_India_vs_Pakistan, T20_Score_Prediction_Factors, T20_World_Cup_2024_Stats

Q2: What pitch type produces the highest scores in Test cricket?
-------------------------------------------------------
Answer : The pitch type that produces the highest scores in Test cricket is a flat dry pitch, typically found in India and Pakistan. According to [Test_Match_Scoring_Patterns], teams expect 350+ in the first innings on batting-friendly surfaces. This is due to the fact that these pitches tend to aid spin bowling later in the match, allowing for extended periods of scoring opportunities.
Sources: Pitch_Report_Types, Virat_Kohli_

---

## Summary

You have built a complete **offline Cricket Score Prediction AI Assistant** using:

| Component | Technology |
|-----------|------------|
| **LLM** | Mistral (via Ollama — no API key!) |
| **Embeddings** | `sentence-transformers/all-MiniLM-L6-v2` |
| **Vector DB** | FAISS (Facebook AI Similarity Search) |
| **Framework** | LangChain |
| **UI** | Gradio |
| **Domain** | Cricket Score Prediction & Analysis |

### Ways to extend this:
- Load real IPL ball-by-ball CSV data from Kaggle for richer factual grounding
- Add live match data via a cricket API (CricAPI, Cricbuzz API)
- Increase chunk size (600-800) for longer match reports and articles
- Swap Mistral for `llama3.2:3b` for improved reasoning quality
- Combine this RAG system with a numerical XGBoost model for quantitative score predictions
- Add a conversational memory buffer to support multi-turn chat history